In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_csv('weekly_driving_profiles.csv')

In [7]:
cols = ['clear_weather','weather_wind_speed_mean','weather_visibility_mean','forward_collision', 'distracted_driver','too_close_distance','lane_departure','driver_making_calls','driver_smoking','fatigue_driving']

df = df.drop(columns=cols)

In [1]:
import os
import wandb # для логирования

import numpy as np
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim # для оптимизаторов
from torchvision import datasets # для данных
import torchvision.transforms as transforms # для преобразований тензоров


In [2]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: masksasha (masksasha-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']
  input_dim = len(feature_cols)
  hidden_dims = [32, 16, 8]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60

hidden_dims = [32, 16, 8] - была совершена проверка (сравнение) разного количества нейронов на 1 слое

Каждый следующий шаг (слой) обобщает информацию (уменьшение нейронов), находии общие зависимости

Adam оптимизатор (по умолчанию) работает лучше всего с lr=0.001 - базовый выбор

60 эпох - практическая проверка. Лучшее обучение на более ранней эпохе, дальше уже переобучение начинается



In [9]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']
target_col = 'claims_count'

unique_drivers = df['driver_id'].unique()

#примерно 70% обучение 15% тест 15% валидация
train_drivers, mr_drivers = train_test_split(unique_drivers, test_size=0.3, random_state=30)
test_drivers, val_drivers = train_test_split( mr_drivers,test_size=0.5, random_state=30)


X_train = df[df['driver_id'].isin(train_drivers)][feature_cols]
X_test = df[df['driver_id'].isin(test_drivers)][feature_cols]
X_val = df[df['driver_id'].isin(val_drivers)][feature_cols]
y_train = df[df['driver_id'].isin(train_drivers)][target_col]
y_test = df[df['driver_id'].isin(test_drivers)][target_col]
y_val = df[df['driver_id'].isin(val_drivers)][target_col]

print("Количество водителей в тренировке: ",len(train_drivers), "Количество строк: ",len(X_train))
print("Количество водителей в тесте: ",len(test_drivers), "Количество строк: ",len(X_test))
print("Количество водителей в валидации: ",len(val_drivers), "Количество строк: ",len(X_val))

#масштабирование признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)

train_data = TensorDataset(X_train_t, y_train_t)
test_data = TensorDataset(X_test_t, y_test_t)
val_data = TensorDataset(X_val_t, y_val_t)

#рекомендуют для датасета 10к-100к батчсайз 64
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)



Количество водителей в тренировке:  247 Количество строк:  8465
Количество водителей в тесте:  53 Количество строк:  2027
Количество водителей в валидации:  54 Количество строк:  2036


In [10]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток
seed_everything(30)

In [11]:
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x

3 слоя Linear - базовая архитектура

ReLU() - базовая функция активации

torch.exp(self.net(x)) - экспонента для положительного результата

In [12]:
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель


Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [13]:
# функция потерь
criterion =  nn.PoissonNLLLoss(log_input=False)

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

Для задачи регрессии, где нам важно больше штрафовать за большие ошибки, чем за маленькие хорошо подходит MSE

Adam - базовый хороший оптимизатор


In [14]:
# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    train_poisson_loss = train_loss_sum / len(train_loader.dataset)

    tqdm.write('\nTrain set: Average poisson loss: {:.4f}'.format(
        train_poisson_loss))

    return train_poisson_loss



In [15]:
def validate(model, device, val_loader, criterion):
    model.eval() # переводим модель в режим инференса
    val_loss_sum = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            val_loss_sum += val_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())

    return  val_loss_sum / len(val_loader.dataset)


In [16]:
# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    test_poisson_loss = test_loss_sum / len(test_loader.dataset)

    tqdm.write('Test set: Average poisson loss: {:.4f}'.format(
       test_poisson_loss))

    return test_poisson_loss



In [17]:
run = wandb.init(
    entity="masksasha-hse-university",
    project="insurance-driving-and-damage",
    group="tabular_discount_prediction",
    name="mlp_baseline",
    config={
        "task": "tabular_claims_count_prediction",
        "model_type": "MLP baseline",
        "target": "claims_count",
        "split_type": "driver_id_train_val_test",
        "random_state": 30,
        "input_dim": CFG.input_dim,
        "hidden_dims": CFG.hidden_dims,
        "dropout_rate": CFG.dropout_rate,
        "batch_size": batch_size,
        "learning_rate": CFG.learning_rate,
        "num_epochs": CFG.num_epochs,
        "loss_function": "PoissonNLLLoss",
        "optimizer": "Adam"
    }
)

In [18]:
# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    model = model.to(device)

    test_losses = []
    val_losses = []
    best_val_loss = 100000
    best_epoch = 0

    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи

        train_poisson_loss = train(model, device, train_loader, optimizer, criterion, epoch)
        val_poisson_loss = validate(model, device, val_loader, criterion)
        val_losses.append(val_poisson_loss)

        wandb.log({
            "train_poisson_loss": train_poisson_loss,
            "val_poisson_loss": val_poisson_loss
        }, step=epoch)

        if val_poisson_loss < best_val_loss:
            best_val_loss = val_poisson_loss
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))

    test_poisson_loss = test(model, device, test_loader, criterion)

    wandb.log({
        "test_poisson_loss": test_poisson_loss
    })

    wandb.run.summary["best_epoch"] = best_epoch
    wandb.run.summary["best_val_poisson_loss"] = best_val_loss
    wandb.run.summary["test_poisson_loss"] = test_poisson_loss

    print('BEST EPOCH: ', best_epoch, 'with Test Loss: ', test_poisson_loss)


In [19]:
main(model)


100%|██████████| 133/133 [00:00<00:00, 336.26it/s]



Train set: Average poisson loss: 1.0385


100%|██████████| 133/133 [00:00<00:00, 523.95it/s]



Train set: Average poisson loss: 0.9105


100%|██████████| 133/133 [00:00<00:00, 501.87it/s]


...


Train set: Average poisson loss: 0.3617


100%|██████████| 133/133 [00:00<00:00, 521.79it/s]



Train set: Average poisson loss: 0.3593


100%|██████████| 133/133 [00:00<00:00, 522.98it/s]



Train set: Average poisson loss: 0.3584
Test set: Average poisson loss: 0.5293
BEST EPOCH:  28 with Test Loss:  0.5293035475076804


In [20]:
wandb.finish()

test_poisson_loss,▁
train_poisson_loss,█▇▆▅▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_poisson_loss,█▇▆▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_epoch,28
best_val_poisson_loss,0.45061
test_poisson_loss,0.5293
train_poisson_loss,0.35837
val_poisson_loss,0.46887


Реализована базовая модель полносвязной нейронной сети

Проверим гиперпараметры, выберем лучшие варианты

In [ ]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims =  [128, 64, 32]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))

def validate(model, device, val_loader, criterion):
    model.eval() # переводим модель в режим инференса
    val_loss_sum = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            val_loss_sum += val_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())

    return  val_loss_sum / len(val_loader.dataset)

# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    criterion =  nn.PoissonNLLLoss(log_input=False)

    optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)
    model = model.to(device)
    test_losses = []
    val_losses = []
    best_val_loss = 100000
    best_epoch = 0
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        train(model, device, train_loader, optimizer, criterion, epoch)
        val_losses.append(validate(model, device, val_loader, criterion))

        if val_losses[-1] < best_val_loss:
            best_val_loss = val_losses[-1]
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))
    test_losses = (test(model, device, test_loader, criterion))
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',test_losses)



Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [ ]:
main(model)

100%|██████████| 133/133 [00:00<00:00, 414.40it/s]



Train set: Average loss: 0.4925


100%|██████████| 133/133 [00:00<00:00, 453.50it/s]



Train set: Average loss: 0.4144


100%|██████████| 133/133 [00:00<00:00, 464.86it/s]


...


Train set: Average loss: 0.1838


100%|██████████| 133/133 [00:00<00:00, 409.30it/s]



Train set: Average loss: 0.1773


100%|██████████| 133/133 [00:00<00:00, 441.43it/s]


Train set: Average loss: 0.1801
Test set: Average loss: 0.5748
BEST EPOCH:  5 with Loss:  0.5747748761192942


In [ ]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims =   [64, 32, 16]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))

def validate(model, device, val_loader, criterion):
    model.eval() # переводим модель в режим инференса
    val_loss_sum = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            val_loss_sum += val_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())

    return  val_loss_sum / len(val_loader.dataset)

# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)

def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    criterion =  nn.PoissonNLLLoss(log_input=False)

    optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)
    model = model.to(device)
    test_losses = []
    val_losses = []
    best_val_loss = 100000
    best_epoch = 0
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        train(model, device, train_loader, optimizer, criterion, epoch)
        val_losses.append(validate(model, device, val_loader, criterion))

        if val_losses[-1] < best_val_loss:
            best_val_loss = val_losses[-1]
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))
    test_losses = (test(model, device, test_loader, criterion))
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',test_losses)



Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [ ]:
main(model)

100%|██████████| 133/133 [00:00<00:00, 477.65it/s]



Train set: Average loss: 0.5007


100%|██████████| 133/133 [00:00<00:00, 415.10it/s]



Train set: Average loss: 0.4177


100%|██████████| 133/133 [00:01<00:00, 112.10it/s]


...


Train set: Average loss: 0.2337


100%|██████████| 133/133 [00:00<00:00, 452.18it/s]



Train set: Average loss: 0.2273


100%|██████████| 133/133 [00:00<00:00, 484.96it/s]



Train set: Average loss: 0.2306
Test set: Average loss: 0.5372
BEST EPOCH:  4 with Loss:  0.5372447735741624


In [ ]:
class CFG:

# Задаем параметры нашего эксперимента
  feature_cols = ['engine_capacity', 'road_quality_moderate', 'slope_flat',
                'motorway', 'rural', 'more_than_one_lane', 'congested',
                'speed_limit_mean', 'weather_temperature_mean', 'total_distance',
                'sum_roundabout', 'sum_traffic_signal', 'sum_stop_sign',
                'sum_yield_sign', 'sum_pedestrian_crossing_sign',
                'sum_animal_crossing_sign', 'speeding_serious',
                'harsh_acceleration', 'harsh_braking']

  input_dim = len(feature_cols)
  hidden_dims =   [256, 128, 64]
  dropout_rate = 0.2
  learning_rate = 0.001
  num_epochs = 60
class Regression(nn.Module): # наследуемся от класса nn.Module
    def __init__(self):
        super(Regression,self).__init__()
        # организуем 3 скрытых слоя
        hidden_1 =  CFG.hidden_dims[0]
        hidden_2 = CFG.hidden_dims[1]
        hidden_3 = CFG.hidden_dims[2]
        #
        input_dim = 19
        self.net = torch.nn.Sequential(
                      torch.nn.Linear(input_dim, hidden_1),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_1, hidden_2),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_2, hidden_3),
                      torch.nn.ReLU(),
                      torch.nn.Linear(hidden_3, 1),
                    )

    def forward(self,x):
        x = torch.exp(self.net(x))
        return x
model = Regression()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) # переводим модель на GPU, не получилось, поэтому CPU
print(model) # посмотрим на нашу модель

# функция потерь
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

# функция обучения модели
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train() # обязательно переводим в режим обучения
    train_loss_sum = 0


    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device) # переводим картинки и таргеты на GPU
        # обнуляем градиенты!
        optimizer.zero_grad()
        # прямой проход
        output = model(data)

        train_loss = criterion(output, target) # считаем значение функции потерь
        # обратный проход
        train_loss.backward()
        # делаем градиентный шаг оптимизатором
        optimizer.step()
        # считаем метрики и лосс
        train_loss_sum += train_loss.item() * data.size(0)

    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(
        train_loss_sum / len(train_loader.dataset)))

def validate(model, device, val_loader, criterion):
    model.eval() # переводим модель в режим инференса
    val_loss_sum = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            val_loss_sum += val_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())

    return  val_loss_sum / len(val_loader.dataset)

# функция тестирования
def test(model, device, test_loader, criterion):
    model.eval() # переводем модель в режим инференса
    test_loss_sum = 0

    all_predictions = []
    all_targets = []

    # показываем, что обученич нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target) # считаем значение функции потерь
            test_loss_sum += test_loss.item() * data.size(0)

            predictions = output.cpu().numpy().flatten()
            targets = target.cpu().numpy().flatten()

            all_predictions.extend(predictions.tolist())
            all_targets.extend(targets.tolist())


            # считаем метрики
    tqdm.write('Test set: Average loss: {:.4f}'.format(
       test_loss_sum / len(test_loader.dataset)))
    return test_loss_sum / len(test_loader.dataset)

# основная функция для экспериментов
def main(model):

    seed_everything(30)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # выделили устройство
    criterion =  nn.PoissonNLLLoss(log_input=False)

    optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)
    model = model.to(device)
    test_losses = []
    val_losses = []
    best_val_loss = 100000
    best_epoch = 0
    for epoch in range(1, CFG.num_epochs + 1): # цикл на эпохи
        train(model, device, train_loader, optimizer, criterion, epoch)
        val_losses.append(validate(model, device, val_loader, criterion))

        if val_losses[-1] < best_val_loss:
            best_val_loss = val_losses[-1]
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model.pth')

    model.load_state_dict(torch.load('best_model.pth'))
    test_losses = (test(model, device, test_loader, criterion))
    print('BEST EPOCH: ', best_epoch, 'with Loss: ',test_losses)



Regression(
  (net): Sequential(
    (0): Linear(in_features=19, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [ ]:
main(model)

100%|██████████| 133/133 [00:00<00:00, 367.21it/s]



Train set: Average loss: 0.4536


100%|██████████| 133/133 [00:00<00:00, 371.26it/s]



Train set: Average loss: 0.4049


100%|██████████| 133/133 [00:00<00:00, 378.10it/s]


...


Train set: Average loss: 0.1661


100%|██████████| 133/133 [00:00<00:00, 341.82it/s]



Train set: Average loss: 0.1707


100%|██████████| 133/133 [00:00<00:00, 339.80it/s]



Train set: Average loss: 0.1691
Test set: Average loss: 0.5810
BEST EPOCH:  3 with Loss:  0.5809751515192649


Вывод:

32 нейрона:
BEST EPOCH:  28 with Loss:  0.5293035475076804

64 нейрона:
BEST EPOCH:  4 with Loss:  0.5372447735741624

128 нейронов:
BEST EPOCH:  5 with Loss:  0.5747748761192942

256 нейронов:
BEST EPOCH:  3 with Loss:  0.5809751515192649

Лучший результат у 32 нейронов. При этом можно отметить скорость обучения - 32 нейрона лучше всего показали себя только на 28 эпохе, а 64/128/256 на 4/5/3 эпохе, что может является показателем переобучения